# NumPy Attributes: Examples of Every Attribute
Basics to advanced. Attributes are accessed **without parentheses** (`a.shape`), unlike methods (`a.reshape()`).

**Sections**
1. Core `ndarray` attributes
2. Memory layout attributes
3. `flags`
4. Views, copies & `base`
5. `dtype` attributes
6. Structured arrays
7. Complex numbers: `real` & `imag`
8. Iterators: `flat`
9. Masked arrays
10. Module constants
11. `finfo` & `iinfo`
12. Ufunc attributes
13. Index helper objects (`r_`, `c_`, `s_`, `mgrid`, `ogrid`)

In [ ]:
import numpy as np
print("NumPy version:", np.__version__)

a = np.array([[1, 2, 3, 4],
              [5, 6, 7, 8],
              [9, 10, 11, 12]])
a

## 1. Core `ndarray` Attributes
| Attribute | Meaning |
|---|---|
| `shape` | tuple of dimension sizes |
| `ndim` | number of dimensions |
| `size` | total number of elements |
| `dtype` | data type of elements |
| `T` | transpose |
| `mT` | matrix transpose of last 2 axes (NumPy 2.0+) |
| `itemsize` | bytes per element |
| `nbytes` | total bytes (`size * itemsize`) |

In [ ]:
print("shape   :", a.shape)
print("ndim    :", a.ndim)
print("size    :", a.size)
print("dtype   :", a.dtype)
print("itemsize:", a.itemsize, "bytes")
print("nbytes  :", a.nbytes, "bytes")
print("len(a)  :", len(a), "(size of first axis)")

In [ ]:
print("T:\n", a.T)
print("T shape:", a.T.shape)

In [ ]:
# mT transposes only the LAST two axes (NumPy 2.0+). Useful for stacked matrices.
b = np.arange(24).reshape(2, 3, 4)
print("b.shape  :", b.shape)
print("b.T.shape:", b.T.shape, "(reverses ALL axes)")
if hasattr(b, "mT"):
    print("b.mT.shape:", b.mT.shape, "(swaps last two only)")
else:
    print("mT needs NumPy 2.0+")

In [ ]:
# Setting shape / dtype-related attributes in place is discouraged; use reshape / astype
c = np.arange(6)
print("1-D :", c.shape)
print("2-D :", c.reshape(2, 3).shape)
print("3-D :", c.reshape(1, 2, 3).shape)
print("0-D :", np.array(5).shape, "| ndim:", np.array(5).ndim)

## 2. Memory Layout Attributes
| Attribute | Meaning |
|---|---|
| `strides` | bytes to step along each axis |
| `data` | memoryview of the buffer |
| `ctypes` | ctypes interface to the buffer |
| `__array_interface__` | dict describing the array buffer |

In [ ]:
print("a.strides  :", a.strides, "-> next row = 32 bytes, next col = 8 bytes")
print("a.T.strides:", a.T.strides, "(transpose just swaps strides, no copy)")

x = np.arange(10, dtype=np.int32)
print("int32 strides:", x.strides)
print("every 2nd    :", x[::2].strides)

In [ ]:
print("data     :", a.data)
print("data type:", type(a.data).__name__)
print("ctypes.data (memory address):", a.ctypes.data)
print("ctypes.shape  :", tuple(a.ctypes.shape[:a.ndim]))
print("ctypes.strides:", tuple(a.ctypes.strides[:a.ndim]))

In [ ]:
info = a.__array_interface__
print("typestr:", info["typestr"])
print("shape  :", info["shape"])
print("strides:", info["strides"])
print("version:", info["version"])

## 3. `flags`
Information about the memory layout and ownership of the array.

| Flag | Meaning |
|---|---|
| `C_CONTIGUOUS` | row-major contiguous |
| `F_CONTIGUOUS` | column-major contiguous |
| `OWNDATA` | array owns its memory |
| `WRITEABLE` | can be modified |
| `ALIGNED` | aligned for the CPU |
| `WRITEBACKIFCOPY` | temporary copy to be written back |

In [ ]:
print(a.flags)

In [ ]:
print("C_CONTIGUOUS:", a.flags.c_contiguous)
print("F_CONTIGUOUS:", a.flags.f_contiguous)
print("OWNDATA     :", a.flags.owndata)
print("WRITEABLE   :", a.flags.writeable)
print("ALIGNED     :", a.flags.aligned)

print("\nTranspose -> C:", a.T.flags.c_contiguous, "| F:", a.T.flags.f_contiguous)
print("Fortran order array -> F:", np.asfortranarray(a).flags.f_contiguous)

In [ ]:
# Make an array read-only
ro = np.array([1, 2, 3])
ro.flags.writeable = False
try:
    ro[0] = 99
except ValueError as e:
    print("ValueError:", e)

## 4. Views, Copies & `base`
`base` is the object that owns the memory (`None` if the array owns its data).

In [ ]:
orig = np.arange(10)
view = orig[2:6]           # slicing -> view (shares memory)
copy = orig[2:6].copy()    # explicit copy

print("orig.base :", orig.base)
print("view.base is orig :", view.base is orig)
print("copy.base :", copy.base)

view[0] = 999
print("orig after changing view:", orig)
print("shares memory:", np.shares_memory(orig, view), "| copy:", np.shares_memory(orig, copy))

In [ ]:
r = a.reshape(4, 3)
print("reshape is a view :", r.base is a)
print("ravel is a view   :", a.ravel().base is a)
print("flatten is a copy :", a.flatten().base is None)
print("fancy index copy  :", a[[0, 2]].base is None)

## 5. `dtype` Attributes
| Attribute | Meaning |
|---|---|
| `name` | readable name |
| `kind` | one-letter type kind |
| `char` | type character code |
| `str` | array-protocol string |
| `itemsize` | bytes per element |
| `byteorder` | `=` native, `<` little, `>` big, `\|` not applicable |
| `type` | scalar Python type |
| `alignment` | alignment in bytes |
| `isnative` | native byte order? |
| `names` / `fields` | field info for structured dtypes |
| `shape` / `subdtype` | sub-array info |
| `hasobject` | contains Python objects? |

In [ ]:
for arr in [np.array([1, 2]), np.array([1.5]), np.array([True]),
            np.array(["hi"]), np.array([1 + 2j]), np.array([None], dtype=object)]:
    d = arr.dtype
    print(f"{str(d):8} name={d.name:10} kind={d.kind} char={d.char} str={d.str:6} "
          f"itemsize={d.itemsize:<3} byteorder={d.byteorder} type={d.type.__name__}")

In [ ]:
d = np.dtype("int32")
print("alignment :", d.alignment)
print("isnative  :", d.isnative)
print("hasobject :", d.hasobject, "| object dtype ->", np.dtype(object).hasobject)
print("isbuiltin :", d.isbuiltin)
print("num       :", d.num)
print("flags     :", d.flags)
print("big-endian int32 byteorder:", np.dtype(">i4").byteorder)
print("little-endian int32 byteorder:", np.dtype("<i4").byteorder)

In [ ]:
# Kind codes
kinds = {"b": "boolean", "i": "signed int", "u": "unsigned int", "f": "float",
         "c": "complex", "m": "timedelta", "M": "datetime",
         "O": "object", "S": "bytes", "U": "unicode", "V": "void/structured"}
for k in ["i", "u", "f", "c", "U"]:
    print(k, "->", kinds[k])

print(np.array(["2024-01-01"], dtype="datetime64[D]").dtype.kind)

## 6. Structured Arrays
Arrays with named fields. Unlocks `dtype.names`, `dtype.fields` and field access.

In [ ]:
people = np.array([("Asha", 25, 50000.0), ("Ravi", 32, 72000.5)],
                  dtype=[("name", "U10"), ("age", "i4"), ("salary", "f8")])
print(people)
print("dtype        :", people.dtype)
print("dtype.names  :", people.dtype.names)
print("dtype.fields :", people.dtype.fields)
print("itemsize     :", people.dtype.itemsize)
print("names column :", people["name"])
print("ages         :", people["age"], "| mean:", people["age"].mean())

In [ ]:
# Sub-array dtype: shape / subdtype
sub = np.dtype((np.int32, (2, 2)))
print("shape   :", sub.shape)
print("subdtype:", sub.subdtype)
print("base    :", sub.base)

## 7. Complex Numbers: `real` and `imag`

In [ ]:
z = np.array([1 + 2j, 3 - 4j, 5j])
print("real:", z.real)
print("imag:", z.imag)
print("dtype:", z.dtype)

# real and imag are views, so editing them edits the array
z.real[0] = 100
print("after editing real:", z)

# For real arrays, imag is zeros (read-only)
r = np.array([1.0, 2.0])
print("real array imag:", r.imag)

## 8. Iterators: `flat`
`flat` is a 1-D iterator (`numpy.flatiter`) over the array.

In [ ]:
print("type:", type(a.flat).__name__)
print("list :", list(a.flat))
print("a.flat[5]     :", a.flat[5])
print("a.flat[2:7]   :", a.flat[2:7])
print("a.flat[[0, 11]]:", a.flat[[0, 11]])

it = a.flat
next(it); next(it)
print("flatiter.index :", it.index)
print("flatiter.coords:", it.coords)
print("flatiter.base is a:", it.base is a)

m = a.copy()
m.flat[::2] = 0       # assign through flat
print(m)

## 9. Masked Arrays (`numpy.ma`)
Arrays with hidden/invalid values. Attributes: `data`, `mask`, `fill_value`, `hardmask`.

In [ ]:
ma = np.ma.array([1, 2, -999, 4, -999], mask=[0, 0, 1, 0, 1])
print("masked array:", ma)
print("data       :", ma.data)
print("mask       :", ma.mask)
print("fill_value :", ma.fill_value)
print("hardmask   :", ma.hardmask)
print("mean       :", ma.mean(), "(ignores masked)")
print("filled(0)  :", ma.filled(0))
print("count      :", ma.count())
print("recordmask :", ma.recordmask)

v = np.ma.masked_less(np.array([5, -1, 3, -2]), 0)
print(v, "| mask:", v.mask)

## 10. Module-Level Constants & Attributes

In [ ]:
print("np.pi           :", np.pi)
print("np.e            :", np.e)
print("np.euler_gamma  :", np.euler_gamma)
print("np.inf          :", np.inf, "| -np.inf:", -np.inf)
print("np.nan          :", np.nan)
print("np.newaxis      :", np.newaxis)
print("np.__version__  :", np.__version__)

print("\nnan == nan      :", np.nan == np.nan, "-> use np.isnan")
print("isnan(np.nan)   :", np.isnan(np.nan))
print("inf > 1e308     :", np.inf > 1e308)

# newaxis adds a dimension
v = np.array([1, 2, 3])
print("\nv.shape:", v.shape, "| v[:, np.newaxis].shape:", v[:, np.newaxis].shape,
      "| v[np.newaxis, :].shape:", v[np.newaxis, :].shape)

## 11. `finfo` & `iinfo`: Limits of Numeric Types
| `finfo` (floats) | `iinfo` (integers) |
|---|---|
| `bits`, `eps`, `max`, `min` | `bits`, `max`, `min` |
| `tiny` / `smallest_normal` | `dtype` |
| `precision`, `resolution` | |
| `dtype` | |

In [ ]:
for t in [np.float16, np.float32, np.float64]:
    f = np.finfo(t)
    print(f"{f.dtype}: bits={f.bits} eps={f.eps:.3e} max={f.max:.3e} "
          f"min={f.min:.3e} smallest_normal={f.smallest_normal:.3e} "
          f"precision={f.precision} resolution={f.resolution}")

In [ ]:
for t in [np.int8, np.int16, np.int32, np.int64, np.uint8]:
    i = np.iinfo(t)
    print(f"{i.dtype}: bits={i.bits} min={i.min} max={i.max}")

# Overflow demo
print("\nuint8 255 + 1 ->", np.array([255], dtype=np.uint8) + np.uint8(1))

## 12. Ufunc Attributes
Universal functions (`np.add`, `np.sin`, ...) are objects with attributes.

| Attribute | Meaning |
|---|---|
| `nin` / `nout` / `nargs` | number of inputs / outputs / total args |
| `ntypes` / `types` | supported type signatures |
| `identity` | identity element (e.g. 0 for add) |
| `signature` | core signature for generalized ufuncs |
| `__name__` | function name |

In [ ]:
for uf in [np.add, np.multiply, np.sin, np.divmod, np.maximum]:
    print(f"{uf.__name__:9} nin={uf.nin} nout={uf.nout} nargs={uf.nargs} "
          f"ntypes={uf.ntypes} identity={uf.identity}")

print("\nnp.add.types[:5]:", np.add.types[:5])
print("np.matmul.signature:", np.matmul.signature)
print("np.add.signature   :", np.add.signature)

## 13. Index Helper Objects
`np.r_`, `np.c_`, `np.s_`, `np.index_exp`, `np.mgrid`, `np.ogrid`, `np.ix_`.

In [ ]:
print("r_ :", np.r_[1:4, 10, 20])          # concatenate along first axis
print("r_ :", np.r_[0:1:5j])                # 5 points from 0 to 1 (complex step = count)
print("c_ :\n", np.c_[[1, 2, 3], [4, 5, 6]])  # column stack

print("\ns_ :", np.s_[1:3, ::2])            # build slice objects
print("a[np.s_[0:2, 1:3]]:\n", a[np.s_[0:2, 1:3]])
print("index_exp:", np.index_exp[1:3, ::2])

In [ ]:
gx, gy = np.mgrid[0:3, 0:3]
print("mgrid dense:\n", gx, "\n", gy)

ox, oy = np.ogrid[0:3, 0:3]
print("\nogrid (open) shapes:", ox.shape, oy.shape)

rows, cols = np.ix_([0, 2], [1, 3])
print("\nix_ selection:\n", a[rows, cols])

## Summary
- **Shape & size:** `shape`, `ndim`, `size`, `len()`
- **Type & memory:** `dtype`, `itemsize`, `nbytes`, `strides`, `data`, `ctypes`
- **Transpose:** `T`, `mT`
- **Layout & ownership:** `flags`, `base`
- **dtype info:** `name`, `kind`, `char`, `str`, `byteorder`, `names`, `fields`, `subdtype`
- **Complex & iteration:** `real`, `imag`, `flat`
- **Masked arrays:** `data`, `mask`, `fill_value`
- **Constants & limits:** `pi`, `e`, `inf`, `nan`, `newaxis`, `finfo`, `iinfo`
- **Ufuncs:** `nin`, `nout`, `types`, `identity`, `signature`